In [6]:
# ─────────────────────────────────────────────────────────────────────────────
#  WhatNet  –  Thai Script Recognition  (PyTorch, folder-based dataset)
#
#  Dataset layout expected:
#    thai-dataset/
#      train/
#        00-161-A1-KO KAI/        ← folder name = class label
#          img001.png
#          ...
#        01-162-A2-KHO KHAI/
#          ...
#      test/
#        00-161-A1-KO KAI/
#          ...
#
#  Labels are derived from folder sort order (0-indexed).
# ─────────────────────────────────────────────────────────────────────────────
import os, time, random, json, math
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms
from PIL import Image

# ─────────────────────────────────────────────────────────────────────────────
#  0. REPRODUCIBILITY
# ─────────────────────────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"[INFO] Using device: {DEVICE}")

# ─────────────────────────────────────────────────────────────────────────────
#  1. CONFIGURATION
# ─────────────────────────────────────────────────────────────────────────────
CFG = {
    "num_classes":     68,       # None = auto-detect
    "image_size":      64,
    "batch_size":      64,
    "epochs":          100,
    "lr":              5e-4,
    "weight_decay":    1e-4,
    "label_smoothing": 0.1,
    "val_split":       0.1,

    "data_dir":    "/teamspace/studios/this_studio/thai-dataset/",
    "results_dir": "./results",
}
os.makedirs(CFG["results_dir"], exist_ok=True)

IMG = CFG["image_size"]
BS  = CFG["batch_size"]

# ─────────────────────────────────────────────────────────────────────────────
#  2. DATASET
# ─────────────────────────────────────────────────────────────────────────────
train_dir = os.path.join(CFG["data_dir"], "train")
test_dir  = os.path.join(CFG["data_dir"], "test")

for d in (train_dir, test_dir):
    if not os.path.isdir(d):
        raise FileNotFoundError(
            f"[ERROR] Expected directory not found: {d}\n"
            f"Make sure CFG['data_dir'] points to the folder that contains "
            f"'train/' and 'test/' sub-directories."
        )

_train_classes = sorted([
    c for c in os.listdir(train_dir)
    if os.path.isdir(os.path.join(train_dir, c))
])
_test_classes = sorted([
    c for c in os.listdir(test_dir)
    if os.path.isdir(os.path.join(test_dir, c))
])

if CFG["num_classes"] is None:
    NUM_CLASSES = len(_train_classes)
    print(f"[INFO] Auto-detected {NUM_CLASSES} classes from train folder.")
else:
    NUM_CLASSES = CFG["num_classes"]
    print(f"[INFO] Using CFG num_classes = {NUM_CLASSES}.")

if len(_train_classes) != len(_test_classes):
    print(f"[WARN] train has {len(_train_classes)} class folders but "
          f"test has {len(_test_classes)} – labels may be inconsistent.")

if len(_train_classes) != NUM_CLASSES:
    print(f"[WARN] Folder count ({len(_train_classes)}) != NUM_CLASSES "
          f"({NUM_CLASSES}). Updating NUM_CLASSES.")
    NUM_CLASSES = len(_train_classes)

print(f"[INFO] Classes (first 5): {_train_classes[:5]} …")

IMG_EXTENSIONS = {".png", ".jpg", ".jpeg", ".bmp", ".tif", ".tiff"}


class ThaiScriptDataset(Dataset):
    """
    Loads greyscale images from a folder hierarchy:
        root/<class_name>/<image_file>

    class_names must be the same sorted list for both train and test splits
    so that class indices are consistent.
    """

    def __init__(self, root: str, class_names: list, transform=None):
        self.transform    = transform
        self.class_to_idx = {c: i for i, c in enumerate(class_names)}
        self.samples      = []   # list of (filepath, label_int)

        for cls in class_names:
            cls_dir = os.path.join(root, cls)
            if not os.path.isdir(cls_dir):
                print(f"[WARN] Missing class folder: {cls_dir}")
                continue
            idx = self.class_to_idx[cls]
            for fname in sorted(os.listdir(cls_dir)):
                fpath = os.path.join(cls_dir, fname)
                if not os.path.isfile(fpath):
                    continue
                if os.path.splitext(fname)[1].lower() not in IMG_EXTENSIONS:
                    continue
                self.samples.append((fpath, idx))

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, index):
        fpath, label = self.samples[index]
        try:
            img = Image.open(fpath).convert("L")   # greyscale
        except Exception as e:
            print(f"[WARN] Could not read {fpath}: {e}")
            img = Image.new("L", (IMG, IMG))
        if self.transform:
            img = self.transform(img)
        return img, label


# ── Transforms ────────────────────────────────────────────────────────────────
#  Normalise to [-1, 1]  (matches TF code: img / 127.5 - 1)
_MEAN = (0.5,)
_STD  = (0.5,)

train_transform = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.RandomAffine(
        degrees=8,                          # ±8° rotation
        translate=(4/64, 4/64),             # ±4 px translation on 64-px image
        fill=0,
    ),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),                  # → [0,1], shape (1, H, W)
    transforms.Normalize(_MEAN, _STD),      # → [-1, 1]
])

eval_transform = transforms.Compose([
    transforms.Resize((IMG, IMG)),
    transforms.ToTensor(),
    transforms.Normalize(_MEAN, _STD),
])

# ── Build datasets & loaders ──────────────────────────────────────────────────
print("[INFO] Indexing train images …")
full_train_ds = ThaiScriptDataset(train_dir, _train_classes, transform=train_transform)
test_ds_raw   = ThaiScriptDataset(test_dir,  _train_classes, transform=eval_transform)

n_total = len(full_train_ds)
n_val   = max(1, int(n_total * CFG["val_split"]))
n_train = n_total - n_val

gen = torch.Generator().manual_seed(SEED)
train_ds, val_ds = random_split(full_train_ds, [n_train, n_val], generator=gen)

# Val split was created from train_transform; swap to eval_transform for val
# The cleanest way: wrap with a view that overrides the transform.
class TransformOverride(Dataset):
    def __init__(self, subset, transform):
        self.subset    = subset
        self.transform = transform

    def __len__(self):
        return len(self.subset)

    def __getitem__(self, idx):
        fpath, label = self.subset.dataset.samples[self.subset.indices[idx]]
        try:
            img = Image.open(fpath).convert("L")
        except Exception:
            img = Image.new("L", (IMG, IMG))
        return self.transform(img), label

val_ds = TransformOverride(val_ds, eval_transform)

train_loader = DataLoader(train_ds,    batch_size=BS, shuffle=True,
                          num_workers=2, pin_memory=True, drop_last=True)
val_loader   = DataLoader(val_ds,      batch_size=BS, shuffle=False,
                          num_workers=2, pin_memory=True)
test_loader  = DataLoader(test_ds_raw, batch_size=BS, shuffle=False,
                          num_workers=2, pin_memory=True)

print(f"[INFO] Train: {n_train:,} | Val: {n_val:,} | Test: {len(test_ds_raw):,}")

# ─────────────────────────────────────────────────────────────────────────────
#  3. DISPLAY UTILITIES
# ─────────────────────────────────────────────────────────────────────────────
_COL = {
    "reset":  "\033[0m",  "bold":   "\033[1m",  "cyan":   "\033[96m",
    "yellow": "\033[93m", "green":  "\033[92m",  "red":    "\033[91m",
    "grey":   "\033[90m", "white":  "\033[97m",  "blue":   "\033[94m",
}

def _c(text, *codes):
    prefix = "".join(_COL.get(c, "") for c in codes)
    return f"{prefix}{text}{_COL['reset']}"

def count_params(model: nn.Module):
    trainable     = sum(p.numel() for p in model.parameters() if p.requires_grad)
    non_trainable = sum(p.numel() for p in model.parameters() if not p.requires_grad)
    return trainable, non_trainable

def print_model_summary(model: nn.Module, name: str) -> None:
    W = 62
    trainable, non_trainable = count_params(model)
    total = trainable + non_trainable
    title = f"  {name}  –  Parameter Summary"
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan')}")
    print(_c(f"║{title:<{W}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╦{'═'*23}╦{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Layer':<16}║  {'Type':<21}║  {'Params':>15}  ║", "cyan", "bold"))
    print(_c(f"╠{'═'*18}╬{'═'*23}╬{'═'*18}╣", "cyan"))
    shown = 0
    for lname, module in model.named_modules():
        n_p = sum(p.numel() for p in module.parameters(recurse=False))
        if n_p > 0:
            shown += 1
            if shown <= 20:
                short = lname[-14:] if len(lname) > 14 else lname
                print(f"║  {short:<16}║  {type(module).__name__[:21]:<21}║  {n_p:>15,}  ║")
    if shown > 20:
        print(f"║  {'… (truncated)':<16}║  {'':21}║  {'':>15}  ║")
    print(_c(f"╠{'═'*18}╩{'═'*23}╩{'═'*18}╣", "cyan"))
    print(_c(f"║  {'Trainable params':<38}: {trainable:>18,}  ║", "green"))
    print(_c(f"║  {'Non-trainable params':<38}: {non_trainable:>18,}  ║", "grey"))
    print(_c(f"║  {'Total params':<38}: {total:>18,}  ║", "bold", "white"))
    print(_c(f"╚{'═'*W}╝", "cyan"))

def print_comparison_table(results: dict) -> None:
    W         = 70
    best_name = max(results, key=lambda k: results[k]["test_acc"])
    print(f"\n{_c('╔' + '═'*W + '╗', 'cyan', 'bold')}")
    print(_c(f"║  {'FINAL TEST-SET COMPARISON':<{W-2}}║", "cyan", "bold"))
    print(_c(f"╠{'═'*24}╦{'═'*12}╦{'═'*12}╦{'═'*12}╦{'═'*6}╣", "cyan"))
    print(_c(f"║  {'Model':<22}║{'Params':>11} ║{'Test Acc':>11} ║{'Macro F1':>11} ║{'Loss':>5} ║",
             "cyan", "bold"))
    print(_c(f"╠{'═'*24}╬{'═'*12}╬{'═'*12}╬{'═'*12}╬{'═'*6}╣", "cyan"))
    for name, r in results.items():
        is_best = (name == best_name)
        star    = "★" if is_best else " "
        row = (f"║{star} {name:<22}║{r['params']:>10,} ║"
               f"{r['test_acc']:>10.2f}%║{r['macro_f1']:>10.2f}%║{r['test_loss']:>5.3f} ║")
        print(_c(row, "green", "bold") if is_best else row)
    print(_c(f"╚{'═'*24}╩{'═'*12}╩{'═'*12}╩{'═'*12}╩{'═'*6}╝", "cyan"))
    print(_c(f"\n  ★  Winner: {best_name}  ({results[best_name]['test_acc']:.2f}% test accuracy)\n",
             "green", "bold"))

# ─────────────────────────────────────────────────────────────────────────────
#  4. BUILDING BLOCKS
# ─────────────────────────────────────────────────────────────────────────────

class ResidualBlock(nn.Module):
    def __init__(self, channels: int):
        super().__init__()
        self.conv1 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(channels)
        self.conv2 = nn.Conv2d(channels, channels, 3, padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(channels)

    def forward(self, x):
        residual = x
        x = F.gelu(self.bn1(self.conv1(x)))
        x = self.bn2(self.conv2(x))
        x = F.gelu(x + residual)
        return x


class DenseResBlock(nn.Module):
    """
    3 stacked ResidualBlocks with dense concatenation, then
    a channel-collapse conv + strided depthwise conv for 2× downsampling.
    """
    def __init__(self, in_channels: int, out_channels: int):
        super().__init__()
        # Optional projection when channel dims differ
        if in_channels != out_channels:
            self.proj = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1, bias=False),
                nn.BatchNorm2d(out_channels),
            )
        else:
            self.proj = None

        self.r1 = ResidualBlock(out_channels)
        self.r2 = ResidualBlock(out_channels)
        self.r3 = ResidualBlock(out_channels)

        # Dense concat: 3 × out_channels → out_channels
        self.collapse = nn.Sequential(
            nn.Conv2d(3 * out_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        )
        # Strided DW + PW for 2× spatial downsampling
        self.dw = nn.Conv2d(out_channels, out_channels, 3, stride=2,
                            padding=1, groups=out_channels, bias=False)
        self.pw = nn.Sequential(
            nn.Conv2d(out_channels, out_channels, 1, bias=False),
            nn.BatchNorm2d(out_channels),
        )

    def forward(self, x):
        if self.proj is not None:
            x = F.gelu(self.proj(x))

        r1 = self.r1(x)
        r2 = self.r2(r1)
        r3 = self.r3(r2)

        cat = torch.cat([r1, r2, r3], dim=1)
        out = F.gelu(self.collapse(cat))
        out = F.gelu(self.pw(self.dw(out)))
        return out


class ChannelAttention(nn.Module):
    def __init__(self, channels: int, reduction: int = 8):
        super().__init__()
        self.fc1 = nn.Linear(channels, channels // reduction)
        self.fc2 = nn.Linear(channels // reduction, channels)

    def forward(self, x):
        # x: (B, C, H, W)
        gap  = x.mean(dim=[2, 3])                    # (B, C)
        attn = F.relu(self.fc1(gap))
        attn = torch.sigmoid(self.fc2(attn))         # (B, C)
        return x * attn.unsqueeze(-1).unsqueeze(-1)


class AdaptiveFilterCapsule(nn.Module):
    """
    Generates per-class capsule activations from a global feature vector.
    """
    def __init__(self, in_features: int, num_classes: int, capsule_dim: int = 8):
        super().__init__()
        self.num_classes = num_classes
        self.capsule_dim = capsule_dim
        self.fc1 = nn.Linear(in_features, 256)
        self.fc2 = nn.Linear(256, num_classes * capsule_dim)
        self.bn  = nn.BatchNorm1d(num_classes)

    def forward(self, x):
        # x: (B, in_features)
        B = x.size(0)
        h = F.gelu(self.fc1(x))
        h = self.fc2(h).view(B, self.num_classes, self.capsule_dim)   # (B, K, cap)

        # x repeated for each class, then sliced to capsule_dim
        x_exp = x.unsqueeze(1).expand(B, self.num_classes, -1)        # (B, K, feat)
        x_sliced = x_exp[..., :self.capsule_dim]                       # (B, K, cap)

        caps = (x_sliced * h).sum(dim=-1)                              # (B, K)
        caps = self.bn(caps)
        return caps


class StrokeTopologyModule(nn.Module):
    """
    Multi-scale feature extraction tuned for Thai script strokes.
    Three parallel 5×5 convolutions (standard, dilated-1, dilated-2).
    """
    def __init__(self, in_channels: int, out_features: int):
        super().__init__()
        self.loop    = nn.Conv2d(in_channels, 64, 5, padding=2, bias=False)
        self.fine    = nn.Conv2d(in_channels, 32, 5, padding=2,
                                 dilation=1, bias=False)
        self.context = nn.Conv2d(in_channels, 32, 5, padding=4,
                                 dilation=2, bias=False)
        self.bn      = nn.BatchNorm2d(128)
        self.fc      = nn.Linear(128, out_features)

    def forward(self, x):
        loop    = F.gelu(self.loop(x))
        fine    = F.gelu(self.fine(x))
        context = F.gelu(self.context(x))
        out = torch.cat([loop, fine, context], dim=1)   # (B,128,H,W)
        out = self.bn(out)
        out = out.mean(dim=[2, 3])                       # GAP → (B,128)
        out = F.gelu(self.fc(out))                       # (B, out_features)
        return out


class CrossScaleTransformerBridge(nn.Module):
    """
    Multi-head self-attention over tokens extracted from three encoder scales.
    """
    def __init__(self, in_channels_list: list, dim: int = 256, num_heads: int = 4):
        super().__init__()
        self.projs = nn.ModuleList([
            nn.Linear(c, dim) for c in in_channels_list
        ])
        self.attn = nn.MultiheadAttention(dim, num_heads, batch_first=True)
        self.ln   = nn.LayerNorm(dim)
        self.fc   = nn.Linear(dim, dim)

    def forward(self, *scale_maps):
        # Each scale_map: (B, C, H, W) → GAP → (B, C) → projected (B, dim)
        tokens = []
        for proj, fmap in zip(self.projs, scale_maps):
            gap = fmap.mean(dim=[2, 3])       # (B, C)
            tok = F.gelu(proj(gap))           # (B, dim)
            tokens.append(tok.unsqueeze(1))   # (B, 1, dim)

        seq = torch.cat(tokens, dim=1)                   # (B, 3, dim)
        attn_out, _ = self.attn(seq, seq, seq)
        attn_out = self.ln(attn_out + seq)
        pooled   = attn_out.mean(dim=1)                  # (B, dim)
        pooled   = F.gelu(self.fc(pooled))
        return pooled


# ─────────────────────────────────────────────────────────────────────────────
#  5. MODEL DEFINITION
# ─────────────────────────────────────────────────────────────────────────────
class WhatNetThai(nn.Module):
    def __init__(self, num_classes: int, image_size: int = 64):
        super().__init__()
        K = num_classes

        # ── Stem: triple-path (3×3, 1×5, 5×1) ────────────────────────────
        self.stem_t  = nn.Sequential(
            nn.Conv2d(1, 32, 3, padding=1, bias=False), nn.BatchNorm2d(32))
        self.stem_h  = nn.Sequential(
            nn.Conv2d(1, 32, (1, 5), padding=(0, 2), bias=False), nn.BatchNorm2d(32))
        self.stem_v  = nn.Sequential(
            nn.Conv2d(1, 32, (5, 1), padding=(2, 0), bias=False), nn.BatchNorm2d(32))

        self.stem_ca = ChannelAttention(96, reduction=8)

        self.stem_pw = nn.Sequential(
            nn.Conv2d(96, 96, 1, bias=False), nn.BatchNorm2d(96))

        # ── Encoder ────────────────────────────────────────────────────────
        self.enc1 = DenseResBlock(96,  96)    # 64→32
        self.enc2 = DenseResBlock(96,  192)   # 32→16
        self.enc3 = DenseResBlock(192, 384)   # 16→8

        # ── Cross-scale transformer bridge ─────────────────────────────────
        self.cstb = CrossScaleTransformerBridge([96, 192, 384], dim=256, num_heads=4)

        # ── AFC ─────────────────────────────────────────────────────────────
        self.afc_proj = nn.Linear(384, 256, bias=False)
        self.afc      = AdaptiveFilterCapsule(256, K, capsule_dim=8)

        # ── Stroke Topology Module ──────────────────────────────────────────
        self.stgm = StrokeTopologyModule(384, 256)

        # ── Gating ─────────────────────────────────────────────────────────
        self.gate = nn.Linear(256 + K, 2)    # combined → 2-way soft gate

        # ── Final classifier ────────────────────────────────────────────────
        self.head = nn.Sequential(
            nn.Linear(256 + K, 512),
            nn.LayerNorm(512),
            nn.GELU(),
            nn.Linear(512, K),
        )

    def forward(self, x):
        # Stem
        t    = F.gelu(self.stem_t(x))
        h    = F.gelu(self.stem_h(x))
        v    = F.gelu(self.stem_v(x))
        stem = torch.cat([t, h, v], dim=1)        # (B, 96, H, W)
        stem = self.stem_ca(stem)
        stem = F.gelu(self.stem_pw(stem))

        # Encoder
        enc1 = self.enc1(stem)   # (B, 96,  32, 32)
        enc2 = self.enc2(enc1)   # (B, 192, 16, 16)
        enc3 = self.enc3(enc2)   # (B, 384,  8,  8)

        # CSTB
        cstb_out = self.cstb(enc1, enc2, enc3)     # (B, 256)

        # AFC
        afc_in  = enc3.mean(dim=[2, 3])            # GAP (B, 384)
        afc_in  = F.gelu(self.afc_proj(afc_in))   # (B, 256)
        afc_in  = afc_in + cstb_out
        afc_out = self.afc(afc_in)                 # (B, K)

        # STGM
        stgm_out = self.stgm(enc3)                 # (B, 256)

        # Gate
        combined = torch.cat([stgm_out, afc_out], dim=1)   # (B, 256+K)
        gate     = F.softmax(self.gate(combined), dim=-1)   # (B, 2)
        stgm_scaled = stgm_out * gate[:, 0:1]               # (B, 256)

        # Fuse & classify
        fused = torch.cat([stgm_scaled, afc_out], dim=1)    # (B, 256+K)
        out   = self.head(fused)                             # (B, K)
        return out


# ─────────────────────────────────────────────────────────────────────────────
#  6. LR SCHEDULE  (cosine annealing)
# ─────────────────────────────────────────────────────────────────────────────
class CosineAnnealingLR(torch.optim.lr_scheduler.LambdaLR):
    """Cosine annealing from base_lr to ~0 over total_steps."""
    def __init__(self, optimizer, total_steps: int, base_lr: float, min_lr: float = 1e-6):
        self.total_steps = total_steps
        self.min_lr      = min_lr
        self.base_lr     = base_lr

        def lr_lambda(step):
            cosine = 0.5 * (1.0 + math.cos(math.pi * step / max(1, total_steps)))
            target = base_lr * cosine
            return max(target, min_lr) / base_lr

        super().__init__(optimizer, lr_lambda=lr_lambda)

# ─────────────────────────────────────────────────────────────────────────────
#  7. TRAINING & EVALUATION HELPERS
# ─────────────────────────────────────────────────────────────────────────────
def train_one_epoch(model, loader, optimizer, scheduler, criterion, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(imgs)
        loss   = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        scheduler.step()

        total_loss += loss.item() * imgs.size(0)
        preds       = logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += imgs.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss, correct, total = 0.0, 0, 0
    for imgs, labels in loader:
        imgs, labels = imgs.to(device), labels.to(device)
        logits = model(imgs)
        loss   = criterion(logits, labels)

        total_loss += loss.item() * imgs.size(0)
        preds       = logits.argmax(dim=1)
        correct    += (preds == labels).sum().item()
        total      += imgs.size(0)

    return total_loss / total, correct / total


@torch.no_grad()
def compute_macro_f1(model, loader, num_classes, device) -> float:
    model.eval()
    tp = np.zeros(num_classes)
    fp = np.zeros(num_classes)
    fn = np.zeros(num_classes)

    for imgs, labels in loader:
        imgs = imgs.to(device)
        preds = model(imgs).argmax(dim=1).cpu().numpy()
        lbls  = labels.numpy()
        for c in range(num_classes):
            tp[c] += np.sum((preds == c) & (lbls == c))
            fp[c] += np.sum((preds == c) & (lbls != c))
            fn[c] += np.sum((preds != c) & (lbls == c))

    prec = tp / (tp + fp + 1e-8)
    rec  = tp / (tp + fn + 1e-8)
    f1   = 2 * prec * rec / (prec + rec + 1e-8)
    return float(f1.mean() * 100.0)

# ─────────────────────────────────────────────────────────────────────────────
#  8. MAIN TRAINING LOOP
# ─────────────────────────────────────────────────────────────────────────────
steps_per_epoch = len(train_loader)
total_steps     = steps_per_epoch * CFG["epochs"]

MODELS = {
    "WhatNet-Thai": WhatNetThai(NUM_CLASSES, IMG),
}

print(_c(f"\n{'═'*70}", "cyan"))
print(_c(f"  Starting benchmark: {len(MODELS)} model(s) × {CFG['epochs']} epochs"
         f"  [Thai folder dataset, {NUM_CLASSES} classes]", "bold"))
print(_c(f"{'═'*70}\n", "cyan"))

trained_models = {}
all_histories  = {}
results        = {}

for name, model in MODELS.items():
    model = model.to(DEVICE)
    print_model_summary(model, name)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=CFG["lr"],
        weight_decay=CFG["weight_decay"],
    )
    scheduler = CosineAnnealingLR(optimizer, total_steps, CFG["lr"])
    criterion = nn.CrossEntropyLoss(label_smoothing=CFG["label_smoothing"])

    ckpt_path  = os.path.join(CFG["results_dir"], f"{name}_best.pt")
    best_val   = 0.0
    history    = {"loss": [], "accuracy": [], "val_loss": [], "val_accuracy": []}

    print(f"\n{_c('  ▶ Training:', 'bold', 'cyan')} {_c(name, 'yellow')}")
    BAR_W = 20
    t0    = time.time()
    patience_counter = 0
    PATIENCE = 10

    for epoch in range(CFG["epochs"]):
        ep_start = time.time()

        train_loss, train_acc = train_one_epoch(
            model, train_loader, optimizer, scheduler, criterion, DEVICE)
        val_loss, val_acc = evaluate(
            model, val_loader, criterion, DEVICE)

        history["loss"].append(train_loss)
        history["accuracy"].append(train_acc)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc)

        # ── Save best checkpoint ──
        if val_acc > best_val:
            best_val = val_acc
            torch.save(model.state_dict(), ckpt_path)
            patience_counter = 0
        else:
            patience_counter += 1

        elapsed   = time.time() - ep_start
        ep_num    = epoch + 1
        pct       = ep_num / CFG["epochs"]
        filled    = int(BAR_W * pct)
        bar       = "█" * filled + "░" * (BAR_W - filled)
        current_lr = scheduler.get_last_lr()[0]

        print(
            f"{_c(f'Epoch {ep_num:>3}/{CFG["epochs"]}', 'grey')}  "
            f"{_c(bar, 'cyan')} {_c(f'{pct*100:>5.1f}%', 'yellow')}  "
            f"{_c(f'loss={train_loss:.4f}', 'white')}  "
            f"{_c(f'acc={train_acc*100:.2f}%', 'green')}  "
            f"{_c(f'val_loss={val_loss:.4f}', 'white')}  "
            f"{_c(f'val_acc={val_acc*100:.2f}%', 'yellow' if val_acc < train_acc else 'green')}  "
            f"{_c(f'lr={current_lr:.2e}', 'blue')}  {_c(f'[{elapsed:.1f}s]', 'grey')}"
        )

        # ── Early stopping ──
        if patience_counter >= PATIENCE:
            print(_c(f"\n  Early stopping at epoch {ep_num} (patience={PATIENCE}).", "yellow"))
            break

    elapsed = time.time() - t0
    print(
        f"\n  {_c('✔ Done:', 'green', 'bold')} "
        f"best val acc = {_c(f'{best_val*100:.2f}%', 'green')}  "
        f"wall time = {_c(f'{elapsed:.0f}s', 'grey')}"
    )

    # ── Restore best weights ──
    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    trained_models[name] = model
    all_histories[name]  = history

# ─────────────────────────────────────────────────────────────────────────────
#  9. FINAL TEST-SET EVALUATION
# ─────────────────────────────────────────────────────────────────────────────
criterion_plain = nn.CrossEntropyLoss()

for name, model in trained_models.items():
    test_loss, test_acc_raw = evaluate(model, test_loader, criterion_plain, DEVICE)
    test_acc  = test_acc_raw * 100.0
    macro_f1  = compute_macro_f1(model, test_loader, NUM_CLASSES, DEVICE)
    trainable, _ = count_params(model)
    results[name] = {
        "test_acc":  round(test_acc, 2),
        "macro_f1":  round(macro_f1, 2),
        "params":    trainable,
        "test_loss": round(float(test_loss), 4),
    }

print_comparison_table(results)

# ─────────────────────────────────────────────────────────────────────────────
#  10. PERSIST RESULTS
# ─────────────────────────────────────────────────────────────────────────────
results_path = os.path.join(CFG["results_dir"], "thai_results.json")
with open(results_path, "w") as f:
    json.dump(results, f, indent=2)
print(f"[INFO] Results   → {results_path}")

histories_path = os.path.join(CFG["results_dir"], "thai_histories.json")
with open(histories_path, "w") as f:
    json.dump(
        {n: {k: [float(v) for v in vals] for k, vals in h.items()}
         for n, h in all_histories.items()},
        f, indent=2,
    )
print(f"[INFO] Histories → {histories_path}")
print(_c("\n[DONE] Thai folder-dataset benchmark complete.\n", "green", "bold"))

[INFO] Using device: cuda
[INFO] Using CFG num_classes = 68.
[WARN] train has 6 class folders but test has 68 – labels may be inconsistent.
[WARN] Folder count (6) != NUM_CLASSES (68). Updating NUM_CLASSES.
[INFO] Classes (first 5): ['05-166-A6-KHO RAKHANG', '18-179-B3-NO NEN', '34-195-C3-RO RUA', '37-198-C6-LU', '49-210-D2-SARA AA'] …
[INFO] Indexing train images …
[INFO] Train: 4,404 | Val: 489 | Test: 1,200

══════════════════════════════════════════════════════════════════════
  Starting benchmark: 1 model(s) × 100 epochs  [Thai folder dataset, 6 classes]
══════════════════════════════════════════════════════════════════════


╔══════════════════════════════════════════════════════════════╗
║  WhatNet-Thai  –  Parameter Summary                          ║
╠══════════════════╦═══════════════════════╦══════════════════╣
║  Layer           ║  Type                 ║           Params  ║
╠══════════════════╬═══════════════════════╬══════════════════╣
║  stem_t.0        ║  Conv2d          

Epoch   1/100  ░░░░░░░░░░░░░░░░░░░░   1.0%  loss=1.7672  acc=18.86%  val_loss=1.7365  val_acc=19.43%  lr=5.00e-04  [3.0s]
Epoch   2/100  ░░░░░░░░░░░░░░░░░░░░   2.0%  loss=1.6534  acc=27.05%  val_loss=1.3353  val_acc=47.03%  lr=5.00e-04  [3.0s]
Epoch   3/100  ░░░░░░░░░░░░░░░░░░░░   3.0%  loss=0.7739  acc=82.03%  val_loss=0.8020  val_acc=82.41%  lr=4.99e-04  [3.0s]
Epoch   4/100  ░░░░░░░░░░░░░░░░░░░░   4.0%  loss=0.5353  acc=95.06%  val_loss=0.5300  val_acc=94.48%  lr=4.98e-04  [3.0s]
Epoch   5/100  █░░░░░░░░░░░░░░░░░░░   5.0%  loss=0.4755  acc=97.77%  val_loss=0.7866  val_acc=82.00%  lr=4.97e-04  [2.9s]
Epoch   6/100  █░░░░░░░░░░░░░░░░░░░   6.0%  loss=0.4728  acc=97.77%  val_loss=0.5503  val_acc=94.68%  lr=4.96e-04  [3.0s]
Epoch   7/100  █░░░░░░░░░░░░░░░░░░░   7.0%  loss=0.4582  acc=98.60%  val_loss=0.4615  val_acc=98.36%  lr=4.94e-04  [3.0s]
Epoch   8/100  █░░░░░░░░░░░░░░░░░░░   8.0%  loss=0.4509  acc=98.92%  val_loss=0.4862  val_acc=97.14%  lr=4.92e-04  [2.9s]
Epoch   9/100  █░░░░░░░░